## Setup

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.status()

In [ ]:
using CairoMakie
using Carlo.ResultTools
using CarloAnalysis
using DataFrames
using FFTW
using HDF5
using JLD2
using LinearAlgebra
using StaticArrays

In [ ]:
function generate_spins(jobname, task_no)
    fig = Figure(size=(400, 400))

    task_str = lpad(task_no, 4, "0")
    h5open("../eta-jobs/$jobname.data/task$task_str/run0001.dump.h5") do file
        spins = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/spins")
        )
        spin_xs = map(v -> v[1], spins)
        spin_ys = map(v -> v[2], spins)
        spin_zs = map(v -> v[3], spins)
        Lx, Ly = size(spins)
        fig[1,1] = Axis(fig; title="Spins", backgroundcolor="black")
        strength = vec(spin_zs)
        arrows2d!(1:Lx, 1:Ly, spin_xs, spin_ys, lengthscale=0.5, align=:center, color=strength,
                  colorrange=(-1, 1))
    end

    return fig
end

In [ ]:
function generate_spinks(jobname, task_no, run_no=1)
    fig = Figure(size=(450, 400))

    task_str = lpad(task_no, 4, "0")
    run_str = lpad(run_no, 4, "0")
    h5open("../eta-jobs/$jobname.data/task$task_str/run$run_str.dump.h5") do file
        spins = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/spins")
        )
        spin_xs = fft(map(v -> v[1], spins)) ./ length(spins)
        spin_ys = fft(map(v -> v[2], spins)) ./ length(spins)
        spin_zs = fft(map(v -> v[3], spins)) ./ length(spins)
        spin_mags = abs2.(spin_xs) + abs2.(spin_ys) + abs2.(spin_zs)
        Lx, Ly = size(spins)
        fig[1,1] = ax = Axis(fig; title="Sk Correlations", backgroundcolor="black")
        hm = heatmap!(ax, spin_mags)
        Colorbar(fig[1,2], hm)
    end

    return fig
end

# Verifying Phases

## FM (Random Init)

In [ ]:
results = JobResult("../eta-jobs", "fm")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="S Gamma Expectation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_e, :T, :sk_Γ, [:Lx], results.data; line=true) do sk
    abs.(getindex.(sk, 3))
end
fig[1,2] = ax_η = Axis(fig, title="S Gamma Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_η, :T, :sk_corr_Γ, [:Lx], results.data; line=true) do sk
    real.(getindex.(sk, 3, 3))
end
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Energy vs. T", xlabel="T", ylabel="Energy")
generate_plot!(ax_e, :T, :Energy, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Heat Capacity vs. T", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax_η, :T, :HeatCap, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Chirality vs. T", xlabel="T", ylabel="Chirality")
generate_plot!(ax_e, :T, :Q, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Chirality^2 vs. T", xlabel="T", ylabel="Chirality^2")
generate_plot!(ax_η, :T, :Q2, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :sk_Γ)
nothing

In [ ]:
CairoMakie.activate!()
i = 38
k_pos = (1, 11)

var1 = :sk_corr_half_M
data1 = first.(mctimes[i][:, var1])
var2 = :ηk_corr_M
data2 = mctimes[i][:, var2]
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, real.(data1))
lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in data2])
fig

## FE (Random Init)

In [ ]:
results = JobResult("../eta-jobs", "fe")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Sx Gamma Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_e, :T, :sk_corr_Γ, [:Lx], results.data; line=true) do sk
    real.(getindex.(sk, 1, 1))
end
fig[1,2] = ax_η = Axis(fig, title="Sy Gamma Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_η, :T, :sk_corr_Γ, [:Lx], results.data; line=true) do sk
    real.(getindex.(sk, 2, 2))
end
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(500, 400))

fig[1,1] = ax_e = Axis(fig, title="Sx+Sy Gamma Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_e, :T, :sk_corr_Γ, [:Lx], results.data; line=true) do sk
    real.(getindex.(sk, 1, 1)) .+ real.(getindex.(sk, 2, 2))
end
Legend(fig[1,2], ax_e)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Energy vs. T", xlabel="T", ylabel="Energy")
generate_plot!(ax_e, :T, :Energy, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Heat Capacity vs. T", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax_η, :T, :HeatCap, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Chirality vs. T", xlabel="T", ylabel="Chirality")
generate_plot!(ax_e, :T, :Q, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Chirality^2 vs. T", xlabel="T", ylabel="Chirality^2")
generate_plot!(ax_η, :T, :Q2, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :sk_Γ)
nothing

In [ ]:
CairoMakie.activate!()
i = 38
k_pos = (1, 11)

var1 = :sk_corr_half_M
data1 = first.(mctimes[i][:, var1])
var2 = :ηk_corr_M
data2 = mctimes[i][:, var2]
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, real.(data1))
lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in data2])
fig

## Stripe

In [ ]:
results = JobResult("../eta-jobs", "stripe")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="In Plane S M Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_e, :T, :sk_corr_M, results.data; label="x corr", line=true) do sk
    real(sk[1,1])
end
generate_plot!(ax_e, :T, :sk_corr_M, results.data; label="y corr", line=true) do sk
    real(sk[2,2])
end
fig[1,2] = ax_η = Axis(fig, title="In Plane Sx M2 Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_η, :T, :sk_corr_M2, [:Lx], results.data; line=true) do sk
    real(sk[1,1])
end
generate_plot!(ax_η, :T, :sk_corr_M2, [:Lx], results.data; line=true) do sk
    real(sk[2,2])
end
Legend(fig[1,3], ax_e)
fig

In [ ]:
fig = Figure(size=(500, 400))

fig[1,1] = ax_e = Axis(fig, title="In Plane S M3 Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_e, :T, :sk_corr_M3, results.data; label="x corr", line=true) do sk
    real(sk[1,1])
end
generate_plot!(ax_e, :T, :sk_corr_M3, results.data; label="y corr", line=true) do sk
    real(sk[2,2])
end
Legend(fig[1,2], ax_e)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Energy vs. T", xlabel="T", ylabel="Energy")
generate_plot!(ax_e, :T, :Energy, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Heat Capacity vs. T", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax_η, :T, :HeatCap, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Chirality vs. T", xlabel="T", ylabel="Chirality")
generate_plot!(ax_e, :T, :Q, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Chirality^2 vs. T", xlabel="T", ylabel="Chirality^2")
generate_plot!(ax_η, :T, :Q2, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :sk_Γ)
nothing

In [ ]:
CairoMakie.activate!()
i = 38
k_pos = (1, 11)

var1 = :sk_corr_half_M
data1 = first.(mctimes[i][:, var1])
var2 = :ηk_corr_M
data2 = mctimes[i][:, var2]
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, real.(data1))
lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in data2])
fig

## Stripe (Negative J+)

In [ ]:
results = JobResult("../eta-jobs", "neg_jp")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="In Plane S M Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_e, :T, :sk_corr_M, results.data; label="x corr", line=true) do sk
    real(sk[1,1])
end
generate_plot!(ax_e, :T, :sk_corr_M, results.data; label="y corr", line=true) do sk
    real(sk[2,2])
end
fig[1,2] = ax_η = Axis(fig, title="In Plane Sx M2 Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_η, :T, :sk_corr_M2, [:Lx], results.data; line=true) do sk
    real(sk[1,1])
end
generate_plot!(ax_η, :T, :sk_corr_M2, [:Lx], results.data; line=true) do sk
    real(sk[2,2])
end
Legend(fig[1,3], ax_e)
fig

In [ ]:
fig = Figure(size=(500, 400))

fig[1,1] = ax_e = Axis(fig, title="In Plane S M3 Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_e, :T, :sk_corr_M3, results.data; label="x corr", line=true) do sk
    real(sk[1,1])
end
generate_plot!(ax_e, :T, :sk_corr_M3, results.data; label="y corr", line=true) do sk
    real(sk[2,2])
end
Legend(fig[1,2], ax_e)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Energy vs. T", xlabel="T", ylabel="Energy")
generate_plot!(ax_e, :T, :Energy, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Heat Capacity vs. T", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax_η, :T, :HeatCap, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Chirality vs. T", xlabel="T", ylabel="Chirality")
generate_plot!(ax_e, :T, :Q, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Chirality^2 vs. T", xlabel="T", ylabel="Chirality^2")
generate_plot!(ax_η, :T, :Q2, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :sk_Γ)
nothing

In [ ]:
CairoMakie.activate!()
i = 38
k_pos = (1, 11)

var1 = :sk_corr_half_M
data1 = first.(mctimes[i][:, var1])
var2 = :ηk_corr_M
data2 = mctimes[i][:, var2]
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, real.(data1))
lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in data2])
fig

## AFM (Random Init)

In [ ]:
results = JobResult("../eta-jobs", "afm")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Sz K Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_e, :T, :sk_corr_K, [:Lx], results.data; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax_η = Axis(fig, title="Sz M2 Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_η, :T, :sk_corr_M, [:Lx], results.data; line=true) do sk
    real(sk[3,3])
end
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Energy vs. T", xlabel="T", ylabel="Energy")
generate_plot!(ax_e, :T, :Energy, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Heat Capacity vs. T", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax_η, :T, :HeatCap, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Chirality vs. T", xlabel="T", ylabel="Chirality")
generate_plot!(ax_e, :T, :Q, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Chirality^2 vs. T", xlabel="T", ylabel="Chirality^2")
generate_plot!(ax_η, :T, :Q2, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
generate_spins("afm", 1)

In [ ]:
generate_spinks("afm", 1)

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_K, :sk_corr_M2)
nothing

In [ ]:
CairoMakie.activate!()
i = 1

var1 = :sk_corr_K
data1 = mctimes[i][:, var1]
var2 = :sk_corr_M2
data2 = mctimes[i][:, var2]
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, [real(ηk[1,1] + ηk[2,2]) for ηk in data1])
lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in data2])
fig

# Param Sweeps

## XXZ Jz Sweep

In [ ]:
results = JobResult("../eta-jobs", "xxz")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Sz Γ Correlation vs. Jz (L=24)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax1, :Jz, :sk_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax2 = Axis(fig, title="S∥ Γ Correlation vs. Jz (L=24)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax2, :Jz, :sk_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk
    real(sk[1,1] + sk[2,2])
end
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Sz Γ Correlation vs. Jz (L=48)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax1, :Jz, :sk_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 48, :]; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax2 = Axis(fig, title="S∥ Γ Correlation vs. Jz (L=48)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax2, :Jz, :sk_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 48, :]; line=true) do sk
    real(sk[1,1] + sk[2,2])
end
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax = Axis(fig, title="Energy vs. Jz (L=24)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax, :Jz, :Energy, [:T], results.data[results.data[:,:Lx] .== 24, :]; line=true)
fig[1,2] = ax2 = Axis(fig, title="Energy vs. Jz (L=48)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax2, :Jz, :Energy, [:T], results.data[results.data[:,:Lx] .== 48, :]; line=true)
Legend(fig[1,3], ax2)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="RMS Chirality vs. Jz (L=24)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax1, :Jz, :chik_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi
    sqrt(chi)
end
fig[1,2] = ax2 = Axis(fig, title="RMS Chirality vs. Jz (L=48)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax2, :Jz, :chik_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi
    sqrt(chi)
end
fig[1,3] = Legend(fig, ax2)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Single Site Chirality vs. Jz", xlabel="Jz", ylabel="Chirality")
generate_plot!(ax_e, :Jz, :single_Q, [:T], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Single Site Chirality^2 vs. Jz", xlabel="Jz", ylabel="Chirality^2")
generate_plot!(ax_η, :Jz, :single_Q2, [:T], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
generate_spinks("gap", 1)

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :sk_Γ, :chik_corr_Γ)
nothing

In [ ]:
CairoMakie.activate!()
i = 28
k_pos = (1, 11)

var1 = :sk_corr_Γ
data1 = real.(getindex.(mctimes[i][:, var1], 3, 3))
var2 = :chik_corr_Γ
data2 = first.(mctimes[i][:, var2])
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, data1)
lines!(ax2, data2)
fig

## J+ = 0.5 Cut

In [ ]:
results = JobResult("../eta-jobs", "jp5")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Sz Γ Correlation vs. T (L=24)", xlabel="T", ylabel="Sk")
generate_plot!(ax1, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax2 = Axis(fig, title="S∥ M+M'+M'' Correlation vs. T (L=24)", xlabel="T", ylabel="Sk")
generate_plot!(ax2, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk
    real(sk[1,1] + sk[2,2])
end
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Sz Γ Correlation vs. T (L=48)", xlabel="T", ylabel="Sk")
generate_plot!(ax1, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax2 = Axis(fig, title="S∥ M+M'+M'' Correlation vs. T (L=48)", xlabel="T", ylabel="Sk")
generate_plot!(ax2, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do sk
    real(sk[1,1] + sk[2,2])
end
Legend(fig[1,3], ax1)

fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Energy vs. T (L=24)", xlabel="T", ylabel="Sk")
generate_plot!(ax1, :T, :Energy, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true)
fig[1,2] = ax2 = Axis(fig, title="Energy vs. T (L=48)", xlabel="T", ylabel="Sk")
generate_plot!(ax2, :T, :Energy, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true)
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Heat Capacity vs. T (L=24)", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax1, :T, :HeatCap, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true)
fig[1,2] = ax2 = Axis(fig, title="Heat Capacity vs. T (L=48)", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax2, :T, :HeatCap, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true)
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Chirality vs. T (L=24)", xlabel="T", ylabel="")
generate_plot!(ax1, :T, :chik_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi
    real(chi)
end
fig[1,2] = ax2 = Axis(fig, title="Chirality vs. T (L=48)", xlabel="T", ylabel="")
generate_plot!(ax2, :T, :chik_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi
    real(chi)
end
fig[1,3] = Legend(fig, ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Chirality^2 Average vs. T (L=24)", xlabel="T")
generate_plot!(ax1, :T, :chi2avg, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi
    sqrt(chi)
end
fig[1,2] = ax2 = Axis(fig, title="Chirality^2 Average vs. T (L=48)", xlabel="T")
generate_plot!(ax2, :T, :chi2avg, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi
    sqrt(chi)
end
fig[1,3] = Legend(fig, ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Intersite Chirality vs. T (L=24)", xlabel="T", ylabel="")
generate_plot!(ax1, :T, [:chi2avg, :chik_corr_Γ], [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi2, chik
    real(chik) - chi2
end
fig[1,2] = ax2 = Axis(fig, title="Intersite Chirality vs. T (L=48)", xlabel="T", ylabel="")
generate_plot!(ax2, :T, [:chi2avg, :chik_corr_Γ], [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi2, chik
    real(chik) - chi2
end
Legend(fig[1,3], ax2)
fig

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :chik_corr_Γ, :chi2avg)
nothing

In [ ]:
CairoMakie.activate!()
i = 60
k_pos = (1, 11)

var1 = :sk_corr_Γ
data1 = real.(getindex.(mctimes[i][:, var1], 3, 3))
var2 = :chik_corr_Γ
data2 = real.(first.(mctimes[i][:, var2]) .- first.(mctimes[i][:, :chi2avg]))
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="Intersite chi vs. Bin #", xlabel="Bin #", ylabel="Intersite chi")
lines!(ax1, data1)
lines!(ax2, data2)
fig

## J+ = 0.9 Cut

In [ ]:
results = JobResult("../eta-jobs", "jp9")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Sz Γ Correlation vs. T (L=24)", xlabel="T", ylabel="Sk")
generate_plot!(ax1, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax2 = Axis(fig, title="S∥ M+M'+M'' Correlation vs. T (L=24)", xlabel="T", ylabel="Sk")
generate_plot!(ax2, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk
    real(sk[1,1] + sk[2,2])
end
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Sz Γ Correlation vs. T (L=48)", xlabel="T", ylabel="Sk")
generate_plot!(ax1, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax2 = Axis(fig, title="S∥ M+M'+M'' Correlation vs. T (L=48)", xlabel="T", ylabel="Sk")
generate_plot!(ax2, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do sk
    real(sk[1,1] + sk[2,2])
end
Legend(fig[1,3], ax1)

fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Energy vs. T (L=24)", xlabel="T", ylabel="Sk")
generate_plot!(ax1, :T, :Energy, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true)
fig[1,2] = ax2 = Axis(fig, title="Energy vs. T (L=48)", xlabel="T", ylabel="Sk")
generate_plot!(ax2, :T, :Energy, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true)
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Heat Capacity vs. T (L=24)", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax1, :T, :HeatCap, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true)
fig[1,2] = ax2 = Axis(fig, title="Heat Capacity vs. T (L=48)", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax2, :T, :HeatCap, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true)
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Chirality vs. T (L=24)", xlabel="T", ylabel="")
generate_plot!(ax1, :T, :chik_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi
    real(chi)
end
fig[1,2] = ax2 = Axis(fig, title="Chirality vs. T (L=48)", xlabel="T", ylabel="")
generate_plot!(ax2, :T, :chik_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi
    real(chi)
end
fig[1,3] = Legend(fig, ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Chirality^2 Average vs. T (L=24)", xlabel="T")
generate_plot!(ax1, :T, :chi2avg, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi
    sqrt(chi)
end
fig[1,2] = ax2 = Axis(fig, title="Chirality^2 Average vs. T (L=48)", xlabel="T")
generate_plot!(ax2, :T, :chi2avg, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi
    sqrt(chi)
end
fig[1,3] = Legend(fig, ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Intersite Chirality vs. T (L=24)", xlabel="T", ylabel="")
generate_plot!(ax1, :T, [:chi2avg, :chik_corr_Γ], [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi2, chik
    real(chik) - chi2
end
fig[1,2] = ax2 = Axis(fig, title="Intersite Chirality vs. T (L=48)", xlabel="T", ylabel="")
generate_plot!(ax2, :T, [:chi2avg, :chik_corr_Γ], [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi2, chik
    real(chik) - chi2
end
Legend(fig[1,3], ax2)
fig

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :chik_corr_Γ, :chi2avg)
nothing

In [ ]:
CairoMakie.activate!()
i = 40
k_pos = (1, 11)

var1 = :sk_corr_Γ
data1 = real.(getindex.(mctimes[i][:, var1], 3, 3))
var2 = :chik_corr_Γ
data2 = real.(first.(mctimes[i][:, var2]) .- first.(mctimes[i][:, :chi2avg]))
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="Intersite chi vs. Bin #", xlabel="Bin #", ylabel="Intersite chi")
lines!(ax1, data1)
lines!(ax2, data2)
fig

## J+ = 1.1 Cut

In [ ]:
results = JobResult("../eta-jobs", "jp11")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Sz Γ Correlation vs. T (L=24)", xlabel="T", ylabel="Sk")
generate_plot!(ax1, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax2 = Axis(fig, title="S∥ M+M'+M'' Correlation vs. T (L=24)", xlabel="T", ylabel="Sk")
generate_plot!(ax2, :T, [:sk_corr_M,:sk_corr_M2,:sk_corr_M3,], [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk1, sk2, sk3
    sk = sk1 + sk2 + sk3
    real(sk[1,1] + sk[2,2])
end
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Sz Γ Correlation vs. T (L=48)", xlabel="T", ylabel="Sk")
generate_plot!(ax1, :T, :sk_corr_Γ, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax2 = Axis(fig, title="S∥ M Correlation vs. T (L=48)", xlabel="T", ylabel="Sk")
generate_plot!(ax2, :T, [:sk_corr_M,:sk_corr_M2,:sk_corr_M3,], [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk1, sk2, sk3
    sk = sk1 + sk2 + sk3
    real(sk[1,1] + sk[2,2])
end
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Energy vs. T (L=24)", xlabel="T", ylabel="Sk")
generate_plot!(ax1, :T, :Energy, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true)
fig[1,2] = ax2 = Axis(fig, title="Energy vs. T (L=48)", xlabel="T", ylabel="Sk")
generate_plot!(ax2, :T, :Energy, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true)
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Heat Capacity vs. T (L=24)", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax1, :T, :HeatCap, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true)
fig[1,2] = ax2 = Axis(fig, title="Heat Capacity vs. T (L=48)", xlabel="T", ylabel="Heat Capacity")
generate_plot!(ax2, :T, :HeatCap, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true)
Legend(fig[1,3], ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Chirality vs. T (L=24)", xlabel="T", ylabel="")
generate_plot!(ax1, :T, :chik_corrs, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi
    real(chi[1,1])
end
fig[1,2] = ax2 = Axis(fig, title="Chirality vs. T (L=48)", xlabel="T", ylabel="")
generate_plot!(ax2, :T, :chik_corrs, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi
    real(chi[1,1])
end
fig[1,3] = Legend(fig, ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Chirality^2 Average vs. T (L=24)", xlabel="T")
generate_plot!(ax1, :T, :chi2avg, [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi
    sqrt(chi)
end
fig[1,2] = ax2 = Axis(fig, title="Chirality^2 Average vs. T (L=48)", xlabel="T")
generate_plot!(ax2, :T, :chi2avg, [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi
    sqrt(chi)
end
fig[1,3] = Legend(fig, ax1)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax1 = Axis(fig, title="Intersite Chirality vs. T (L=24)", xlabel="T", ylabel="")
generate_plot!(ax1, :T, [:chi2avg, :chik_corrs], [:Jz], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi2, chik
    real(chik[1,1]) - chi2
end
fig[1,2] = ax2 = Axis(fig, title="Intersite Chirality vs. T (L=48)", xlabel="T", ylabel="")
generate_plot!(ax2, :T, [:chi2avg, :chik_corrs], [:Jz], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi2, chik
    real(chik[1,1]) - chi2
end
Legend(fig[1,3], ax1)
fig

In [ ]:
i = 34
fig = Figure(size=(500, 400))
fig[1,1] = ax1 = Axis(fig, title="k-space χ Correlation")

kcorrs = real.(results.data[i, :chik_corrs])
hm = heatmap!(ax1, getfield.(kcorrs, :val))
Colorbar(fig[1,2], hm)

fig

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :chik_corr_Γ)
nothing

In [ ]:
CairoMakie.activate!()
i = 7
k_pos = (1, 11)

var1 = :sk_corr_Γ
data1 = real.(getindex.(mctimes[i][:, var1], 3, 3))
var2 = :chik_corr_Γ
data2 = real.(first.(mctimes[i][:, var2]))
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, data1)
lines!(ax2, data2)
fig

## FE-Stripe Phase Boundary

In [ ]:
results = JobResult("../eta-jobs", "gap")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Sz Γ Correlation vs. Jz (L=24)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax_e, :Jz, :sk_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 24, :]; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax_η = Axis(fig, title="Sz Γ Correlation vs. Jz (L=48)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax_η, :Jz, :sk_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 48, :]; line=true) do sk
    real(getindex(sk, 3, 3))
end
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Energy vs. Jz (L=24)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax_e, :Jz, :Energy, [:T], results.data[results.data[:,:Lx] .== 24, :]; line=true)
fig[1,2] = ax_η = Axis(fig, title="Energy vs. Jz (L=48)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax_η, :Jz, :Energy, [:T], results.data[results.data[:,:Lx] .== 48, :]; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 800))

fig[1,1] = ax1 = Axis(fig, title="RMS Chirality vs. Jz (L=24)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax1, :Jz, :chik_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 24, :]; line=true) do chi
    sqrt(chi)
end
fig[1,2] = ax2 = Axis(fig, title="RMS Chirality vs. Jz (L=48)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax2, :Jz, :chik_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 48, :]; line=true) do chi
    sqrt(chi)
end
fig[2,1] = ax3 = Axis(fig, title="RMS Chirality vs. Jz (L=96)", xlabel="Jz", ylabel="Sk")
generate_plot!(ax3, :Jz, :chik_corr_Γ, [:T], results.data[results.data[:,:Lx] .== 96, :]; line=true) do chi
    sqrt(chi)
end
fig[2,2] = Legend(fig, ax3, tellwidth=false, halign=:left, valign=:top)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Single Site Chirality vs. Jz", xlabel="Jz", ylabel="Chirality")
generate_plot!(ax_e, :Jz, :single_Q, [:T], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Single Site Chirality^2 vs. Jz", xlabel="Jz", ylabel="Chirality^2")
generate_plot!(ax_η, :Jz, :single_Q2, [:T], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
generate_spinks("gap", 1)

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :sk_Γ, :chik_corr_Γ)
nothing

In [ ]:
CairoMakie.activate!()
i = 28
k_pos = (1, 11)

var1 = :sk_corr_Γ
data1 = real.(getindex.(mctimes[i][:, var1], 3, 3))
var2 = :chik_corr_Γ
data2 = first.(mctimes[i][:, var2])
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, data1)
lines!(ax2, data2)
fig

## Triple Point

In [ ]:
results = JobResult("../eta-jobs", "triple-point")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Sz Γ Correlation vs. T", xlabel="T", ylabel="Sk")
generate_plot!(ax_e, :T, :sk_corr_Γ, [:Lx], results.data; line=true) do sk
    real(sk[3,3])
end
fig[1,2] = ax_η = Axis(fig, title="RMS Chirality vs. T", xlabel="T", ylabel="chik")
generate_plot!(ax_η, :T, :chik_corr_Γ, [:Lx], results.data; line=true) do chik
    sqrt(chik)
end
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Energy vs. T", xlabel="T", ylabel="Energy")
generate_plot!(ax_e, :T, :Energy, [:Lx], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Heat Capacity vs. T", xlabel="T", ylabel="HeatCap")
generate_plot!(ax_η, :T, :HeatCap, [:Lx], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
generate_spinks("triple-point", 1)

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :sk_Γ, :chik_corr_Γ)
nothing

In [ ]:
CairoMakie.activate!()
i = 38

var1 = :sk_corr_Γ
data1 = real.(getindex.(mctimes[i][:, var1], 3, 3))
var2 = :chik_corr_Γ
data2 = sqrt.(first.(mctimes[i][:, var2]))
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, data1)
lines!(ax2, data2)
fig

## Jz=0.4 Cut

In [ ]:
results = JobResult("../eta-jobs", "jz_cut")

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="In plane S Γ Correlation vs. Jp", xlabel="Jp", ylabel="Sk")
generate_plot!(ax_e, :Jp, :sk_corr_Γ, [:T], results.data; line=true) do sk
    real(getindex(sk, 1, 1) + getindex(sk, 2, 2))
end
fig[1,2] = ax_η = Axis(fig, title="Sz Γ Correlation vs. Jp", xlabel="Jp", ylabel="Sk")
generate_plot!(ax_η, :Jp, :sk_corr_Γ, [:T], results.data; line=true) do sk
    real(getindex(sk, 3, 3))
end
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="In plane S∥ M+M'+M'' Correlation vs. Jp", xlabel="Jp", ylabel="Sk")
generate_plot!(ax_e, :Jp, [:sk_corr_M, :sk_corr_M2, :sk_corr_M3], [:T], results.data; line=true) do s1, s2, s3
    real(tr(s1) + tr(s2) + tr(s3))
end
fig[1,2] = ax_η = Axis(fig, title="In plane S∥ M2 Correlation vs. Jp", xlabel="Jp", ylabel="Sk")
generate_plot!(ax_η, :Jp, :sk_corr_M, [:T], results.data; line=true) do sk
    real(getindex(sk, 1, 1) + getindex(sk, 2, 2))
end
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(500, 400))

fig[1,1] = ax_e = Axis(fig, title="Sz Magnetization Correlation vs. Jp", xlabel="Jp", ylabel="Sk")
generate_plot!(ax_e, :Jp, :sk_corr_Γ, [:T], results.data; line=true) do sk
    real(sk[3,3])
end
Legend(fig[1,2], ax_e)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Energy vs. Jp", xlabel="Jp", ylabel="Energy")
generate_plot!(ax_e, :Jp, :Energy, [:T], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Heat Capacity vs. Jp", xlabel="Jp", ylabel="Heat Capacity")
generate_plot!(ax_η, :Jp, :HeatCap, [:T], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Chirality vs. Jz", xlabel="Jz", ylabel="Chirality")
generate_plot!(ax_e, :Jp, :Q, [:T], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Chirality^2 vs. Jz", xlabel="Jz", ylabel="Chirality^2")
generate_plot!(ax_η, :Jp, :Q2, [:T], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax_e = Axis(fig, title="Single Site Chirality vs. Jz", xlabel="Jz", ylabel="Chirality")
generate_plot!(ax_e, :Jp, :single_Q, [:T], results.data; line=true)
fig[1,2] = ax_η = Axis(fig, title="Single Site Chirality^2 vs. Jz", xlabel="Jz", ylabel="Chirality^2")
generate_plot!(ax_η, :Jp, :single_Q2, [:T], results.data; line=true)
Legend(fig[1,3], ax_η)
fig

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_Γ, :sk_Γ, :Q)
nothing

In [ ]:
CairoMakie.activate!()
i = 18
k_pos = (1, 11)

var1 = :Q
data1 = first.(mctimes[i][:, var1])
var2 = :Q
data2 = first.(mctimes[i][:, var2])
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
lines!(ax1, data1)
lines!(ax2, data2)
fig